# 第 6 周 · 周日：周复习与阶段二总结——从工具到组合的蜕变

> 定位：回顾W5-W6的SQL分析思维，完成阶段二(数据工程思维)的全面总结，为下半程(推断思维)做好准备
> 今日目标：查漏补缺、构建知识地图、完成自评、规划下半程学习重点

---

## 📘 区块1：W6 本周回顾——实战篇全览

### 1.1 W6每日核心认知回顾

| 日期 | 主题 | 最核心的认知(一句话) | 自评(1-5) |
|------|------|-------------------|----------|
| 周一 | 复杂业务问题分解 | 复杂SQL = 3-5个CTE串起来——先在纸上画数据变换管道 | ___ |
| 周二 | 用户画像SQL | GROUP BY user_id + CASE WHEN = 80%的画像标签 | ___ |
| 周三 | 活动效果评估 | "活动期间GMV涨了"≠活动有效——要排除时间转移和自然增长 | ___ |
| 周四 | 经典分析题+面试 | 8道经典题覆盖90%面试题型——必须能不看笔记默写 | ___ |
| 周五 | SQL性能调优 | 不看EXPLAIN就优化=不看化验单就开药 | ___ |
| 周六 | 综合实战 | 6个SQL查询=1份完整的月度运营报告 | ___ |

**诚实自评指南**：1=完全不会，2=能看懂但写不出，3=能参考笔记写出，4=能独立写出，5=能灵活运用并讲给别人听。分数<3的项目是下周重点补习对象。

### 1.2 W6技能清单完整自查

**初级(必须全部掌握，这是及格线)**：
- [ ] 用CTE链式拆解复杂业务问题(3-5个CTE)
- [ ] 构建基础用户画像表(价值分层+活跃状态+品类偏好)
- [ ] 写出活动前/中/后三段对比的SQL
- [ ] 写出6大经典SQL模式(TopN/留存/连续/累计/漏斗/占比)
- [ ] 使用EXPLAIN诊断慢查询(关注type/rows/Extra)
- [ ] 完成月度运营报告的6个核心查询

**中级(应该掌握80%以上，这是合格分析师的水平)**：
- [ ] RFM三维打分及用户价值分层
- [ ] DID双重差分效果估算
- [ ] 多模式组合的面试综合题解法
- [ ] 数据倾斜的诊断和4种处理方案
- [ ] 渠道获客质量分析(GMV per user)
- [ ] 高质量新用户的多维度定义和筛选

**高级(追求掌握，这是高级分析师的水平)**：
- [ ] 完整画像标签体系设计(5维度×20+标签)
- [ ] 活动ROI的完整计算(含毛利率+全部成本)
- [ ] 大数据场景的完整优化方案(分区+索引+倾斜处理)
- [ ] 分析报告的自动化方案设计(SQL+Cron+BI)
- [ ] 从分析发现到业务行动的完整闭环设计

### 1.3 W5-W6 完整知识地图

```
W5 SQL分析思维(上)——工具篇(学乐器)
├── D1: 分析思维入门(取数vs分析的区别)
├── D2: 窗口函数(ROW_NUMBER/RANK/LAG/SUM OVER) ← 最重要的SQL进阶技能
├── D3: 连续登录与留存(Gap-and-Island)
├── D4: JOIN全家桶(四种JOIN + LEFT JOIN陷阱 + NOT IN陷阱)
├── D5: CTE与子查询(让SQL可读可维护)
└── D6: 三大经典SQL(TopN/复购率/漏斗)

W6 SQL分析思维(下)——实战篇(组建乐队)
├── D1: 复杂业务问题分解(CTE链式设计方法论)
├── D2: 用户画像SQL(GROUP BY + CASE WHEN聚合)
├── D3: 活动效果评估(三段对比 + DID双重差分)
├── D4: 经典分析题训练(8道题 + 面试场景模拟)
├── D5: SQL性能调优(EXPLAIN + 数据倾斜 + 分区索引)
└── D6: 端到端月度报告(6个查询 = 1份完整报告)
```

**W5和W6的关系**：W5=学会了每个乐器(窗口函数=钢琴、JOIN=吉他、CTE=贝斯、子查询=鼓)。W6=学会了组建乐队——把乐器组合起来演奏一首完整的曲子(月度分析报告)。单会乐器不够，单会组乐队也不行——两者缺一不可。

### 1.4 W6最重要的10个SQL技能(自测：看到名字能写出SQL吗？)

1. **CTE链式设计**：WITH a AS (...), b AS (...), c AS (...) SELECT * FROM c
2. **排名+筛选**：ROW_NUMBER() OVER(PARTITION BY x ORDER BY y DESC) AS rn
3. **留存计算**：LEFT JOIN ON a.id=b.id AND b.date=DATE_ADD(a.date, INTERVAL N DAY)
4. **连续行为**：ROW_NUMBER + DATE_SUB(rn) + GROUP BY grp → gap-and-island
5. **累计值**：SUM(x) OVER(PARTITION BY id ORDER BY date) 或加ROWS帧
6. **占比分析**：SUM(x)*100.0 / SUM(SUM(x)) OVER() → 聚合嵌套窗口
7. **漏斗分析**：COUNT(DISTINCT CASE WHEN event='X' THEN user_id END)
8. **用户画像**：GROUP BY user_id + 多个CASE WHEN聚合
9. **活动评估**：SUM(CASE WHEN period THEN amount ELSE 0 END) 三段对比
10. **性能诊断**：EXPLAIN → 看type → 看rows → 看Extra → 优化


## 🧩 区块2：核心技能深度速查

### 2.1 窗口函数速查卡——最重要的SQL进阶技能

```sql
-- 排名三兄弟
ROW_NUMBER() → 1,2,3,4 (强制唯一，严格取N个)
RANK()       → 1,1,3,4 (并列跳跃，适合排行榜)
DENSE_RANK() → 1,1,2,3 (并列连续，适合等级划分)

-- 前后取值
LAG(col, offset, default)  → 向前取(昨天/上月)
LEAD(col, offset, default) → 向后取(明天/下月)

-- 累计/滚动
SUM(col) OVER (PARTITION BY id ORDER BY date)  → 累计
SUM(col) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) → 7天滚动
AVG(col) OVER (ORDER BY date ROWS 29 PRECEDING) → 30天移动平均

-- 分层分位
NTILE(N) OVER (ORDER BY col)        → N等分(按行数均分)
PERCENT_RANK() OVER (ORDER BY col)  → 百分比排名(0-1之间)
```

### 2.2 JOIN陷阱速查表

| 陷阱 | 原因 | 症状 | 解决方法 |
|------|------|------|--------|
| LEFT JOIN + WHERE右表条件 = INNER JOIN | WHERE在JOIN后过滤NULL | 左表行数减少 | 右表条件放ON里 |
| NOT IN子查询有NULL返回空集 | NULL比较=UNKNOWN | 结果意外为空 | 用NOT EXISTS替代 |
| 一对多JOIN行数膨胀 | 左表1行→右表多行匹配 | COUNT(*)变大 | 先聚合右表再JOIN |
| 笛卡尔积(JOIN条件遗漏) | ON缺少关联条件 | 行数=左表×右表 | 检查ON条件完整性 |
| JOIN条件NULL不匹配 | NULL != NULL | 匹配失败 | COALESCE处理或用IS NULL |

### 2.3 Gap-and-Island四步法速查

```sql
-- 步骤1：去重(同一天只保留一条)
SELECT DISTINCT user_id, login_date FROM logins
-- 步骤2：编号(每人按日期排序)
ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn
-- 步骤3：日期减编号→得到分组键(连续的日期会得到相同的grp)
DATE_SUB(login_date, INTERVAL rn DAY) AS grp
-- 步骤4：按user_id+grp分组→COUNT(*)即连续天数
SELECT user_id, grp, MIN(login_date), MAX(login_date), COUNT(*) AS days
FROM t3 GROUP BY user_id, grp
```

### 2.4 CTE设计五原则

1. **一个CTE只做一件事**(过滤/聚合/排名/筛选各自独立)
2. **命名描述"是什么"**(user_monthly_spend)而非"做了什么"(step1/temp)
3. **依赖关系线性排列**(从上到下顺序引用，不交叉)
4. **3-5个CTE最佳**(太少没必要，太多需审视)
5. **可独立测试**(每个CTE都能单独SELECT验证)

### 2.5 性能优化决策树

```
查询很慢！→ 先跑EXPLAIN看type:
  ├── type=ALL(全表扫描) → 检查：有没有分区条件？有没有索引？
  │   ├── 没分区条件 → WHERE里加上分区字段
  │   ├── 没索引 → 在WHERE/JOIN列上建索引
  │   └── 索引失效 → 检查：用了函数？类型转换？LIKE'%x'？
  ├── type=ref但rows很大 → 索引选择性差 → 考虑复合索引或覆盖索引
  ├── Extra=Using filesort → ORDER BY没走索引 → 为ORDER BY列建索引
  ├── Extra=Using temporary → GROUP BY/DISTINCT创建了临时表
  │   → 优化聚合逻辑或增加sort_buffer_size
  └── 大部分task快但个别慢 → 数据倾斜 → 加盐/MapJoin/拆分
```

### 2.6 12个SQL模式速查卡片(面试前一天背诵)

```sql
-- 1. 分组TopN
ROW_NUMBER() OVER(PARTITION BY group_col ORDER BY val DESC) rn ... WHERE rn <= N
-- 2. 次日留存
LEFT JOIN ON a.id=b.id AND b.date = DATE_ADD(a.date, INTERVAL 1 DAY)
-- 3. 连续登录
DISTINCT → ROW_NUMBER OVER(PARTITION BY uid ORDER BY dt) → DATE_SUB(dt, INTERVAL rn DAY) grp → GROUP BY uid,grp
-- 4. 累计值
SUM(col) OVER(PARTITION BY id ORDER BY date ROWS UNBOUNDED PRECEDING)
-- 5. 7天滚动
SUM(col) OVER(ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
-- 6. 品类占比
SUM(col)*100.0 / SUM(SUM(col)) OVER()
-- 7. 漏斗分析
COUNT(DISTINCT CASE WHEN event='stepN' THEN user_id END)
-- 8. 用户分层
CASE WHEN val>=high THEN '高' WHEN val>=mid THEN '中' ELSE '低' END
-- 9. 复购率
GROUP BY user_id → HAVING COUNT(*) >= 2 → COUNT(*)
-- 10. 活动三段对比
SUM(CASE WHEN dt<'start' THEN amt ELSE 0 END) AS before_amt
-- 11. DID双重差分
(AVG(活动组post) - AVG(活动组pre)) - (AVG(对照组post) - AVG(对照组pre))
-- 12. RFM三维打分
R = DATEDIFF(today, MAX(date))   -- 越小越好
F = COUNT(DISTINCT order_id)      -- 越多越好
M = SUM(amount)                   -- 越高越好
```

## 🧩 区块3：阶段一+二总回顾(W1-W6)——你已走完了半程

### 3.1 整个课程的四大阶段

| 阶段 | 周数 | 核心能力 | 标志性技能 | 状态 |
|------|------|---------|----------|------|
| 商业思维 | W1-W3 | 看到指标知道它意味着什么 | 指标拆解、漏斗分析、北极星指标 | ✅ |
| 数据工程 | W4-W6 | 知道数据从哪来、怎么存、怎么高效分析 | SQL从基础到性能优化 | ✅ |
| 推断思维 | W7-W9 | 判断因果而非相关 | AB实验设计、因果推断方法 | → 开始 |
| 整合应用 | W10-W12 | 综合运用所有技能 | 端到端分析项目 | → 即将 |

### 3.2 W1-W6 核心能力自评

**商业思维(W1-W3)**：
- [ ] 看到GMV变化能拆解到因子层面(用户数×转化率×客单价)
- [ ] 能用漏斗分析定位转化率最低的环节
- [ ] 能区分"虚荣指标"(PV/UV)和"可执行指标"(转化率/GMV/留存)
- [ ] 能定义产品的北极星指标并说出理由
- [ ] 能设计一个业务问题的分析框架(从指标定义到数据收集到分析方法)

**数据工程(W4-W6)**：
- [ ] 理解数据仓库的分层架构(ODS/DWD/DWS/DM)
- [ ] 能独立完成从数据提取到分析报告的完整SQL流程
- [ ] 能写出包含窗口函数/CTE/JOIN的复杂SQL(5层CTE链)
- [ ] 能诊断和优化慢查询(EXPLAIN+数据分布+索引策略)
- [ ] 能构建用户画像标签体系(5维度×20+标签)

### 3.3 学习曲线回顾——从"会用"到"精通"的蜕变

```
W1: "原来指标不只是数字——每个指标背后都有业务含义"
W2: "漏斗分析可以这么深入——每个环节的流失都有原因"
W3: "北极星指标不是拍脑袋——是严格的推导过程"
W4: "数据仓库的设计原来这么讲究——分层、分域、建模方法论"
W5: "窗口函数打开了一扇全新的大门——SQL原来这么强大"
W6: "我能独立完成一份完整的分析了——从需求到报告全流程"  ← 你在这里
W7: "不能做实验时，怎么判断是A导致了B？"  → AB实验入门
W8: "AB实验的设计竟然这么严谨——分流/样本量/统计显著性"
W9: "不能做实验怎么办？——因果推断五方法"
W10-12: "把这些技能全部串起来，做一个真正的分析项目"
```

### 3.4 到目前为止最应该能在面试中展示的10项能力

1. 用窗口函数解决分组TopN、累计值、环比同比问题
2. 用CTE链式设计拆解复杂业务分析需求
3. 独立写出次日/7日/30日留存SQL
4. 独立写出连续登录SQL(gap-and-island四步法)
5. 识别和规避JOIN的5大常见陷阱
6. 构建用户画像标签表(价值+活跃+偏好+行为+生命周期)
7. 评估活动效果(三段对比 + DID双重差分估算)
8. 计算漏斗转化率和复购率
9. 使用EXPLAIN诊断慢查询并给出完整优化方案
10. 完成一份从数据到报告的端到端分析(含解读和建议)

**如果能独立完成以上10项，你在SQL方面的能力已经达到大厂中级数据分析师的标准。**
**如果能在此基础上主动讨论边界情况和性能优化，你已达到高级标准。**


## 🧩 区块4：下半程(W7-W12)预告

### 4.1 W7-W8：AB实验——数据分析师的进阶必修课

**W7-W8将解决的核心问题**：
- "活动期间GMV涨了，怎么确定是活动导致的？"(W6D3的进阶——加入随机化分流)
- "实验组比对照组高5%，这个差异是真实效果还是随机波动？"(统计显著性检验：t检验/p值/置信区间)
- "需要多少用户参与实验才能检测到预期的效果？"(样本量计算/统计功效分析)
- "实验期间发生了外部事件(比如竞品大促)怎么办？"(实验质量控制/AA检验/CUPED)
- "如何保证实验组和对照组在所有维度上均衡？"(随机分流/分层抽样/AA验证)

**与01python_learn的同步**：01python_learn W7-W8学统计学(t检验/卡方检验/方差分析/回归)和ML基础——你在那边学统计理论的同时，这边同步学AB实验的设计和分析方法。理论+实践同步推进，效果加倍。

### 4.2 W9：因果推断——不能做实验时的救命稻草

**W9将解决的核心问题**：
- "不能做实验时(比如价格调整、政策变化、产品大改版)，怎么逼近因果结论？"
- 核心方法体系：
  - DID(你已学)——利用时间+人群的双重差异
  - 匹配法(Propensity Score Matching)——为实验组"找相似"的对照组
  - 断点回归(Regression Discontinuity)——利用截断阈值模拟随机化
  - 工具变量(Instrumental Variables)——利用外部变量识别因果
  - 合成控制法(Synthetic Control)——为单个实验对象构造"合成对照"
- 真实案例：Uber动态定价效果评估、Amazon推荐算法增量评估、政策效果评估

**前置依赖**：W6D3的活动效果评估(特别是DID)是W9因果推断的入门基础。你已经有了DID的概念——W9会把它和其他方法串联成一个完整的方法论体系。

### 4.3 W10-W12：整合应用——从学到用的最后一步

**W10-W12的学习模式将完全改变**：
- 不再是"每天学一个新知识点"
- 而是"用W1-W9学的所有知识，完成一个端到端的完整分析项目"
- 你将模拟真实工作场景：接到业务需求→设计方案→取数分析→统计分析→报告呈现→推动决策落地
- 内容包含：案例深度分析、综合项目实战、面试全真模拟、知识体系总复习

### 4.4 上半程和下半程的关系

```
上半程(W1-W6)：建立完整的分析工具箱
  ├── 商业思维(W1-W3)：知道分析什么、怎么判断对错
  ├── 数据工程(W4-W6)：知道数据在哪、怎么高效分析
  └── 结果：能独立完成"描述性分析"(发生了什么？)

下半程(W7-W12)：解决更高级的问题
  ├── AB实验(W7-W8)：能设计实验验证"为什么发生？"
  ├── 因果推断(W9)：不能实验时也能推断因果关系
  └── 整合应用(W10-W12)：把所有技能用于真实场景
  └── 结果：能回答"为什么发生？"和"我们应该怎么做？"
```

**上半程是"描述世界"——看清发生了什么。**
**下半程是"改变世界"——验证怎么做才能变得更好。**

两者结合，才是完整的数据驱动决策能力。


## 💻 区块5：动手自测——W5-W6综合能力检验

请独立完成以下5道综合自测题(不看笔记)，检验你是否真正掌握了W5-W6的核心技能。

**评分标准**：
- 5/5题独立完成 → 优秀！可以直接进入下半程
- 4/5题独立完成 → 良好！个别知识点再巩固一下(标记做错的那1题)
- 3/5题独立完成 → 及格，但建议重点复习不熟的2题对应的W5-W6模块
- <3题独立完成 → 建议W5-W6重点内容再复习1-2天再进入下半程


In [ ]:
import sqlite3, pandas as pd, numpy as np
np.random.seed(123); conn=sqlite3.connect(':memory:')

# 生成自测数据
n=1000
df=pd.DataFrame({
    'user_id':np.random.choice(range(1,101),n),
    'product':np.random.choice(['A','B','C','D'],n),
    'amount':np.round(np.random.lognormal(4,0.6,n),2),
    'dt':np.random.choice(pd.date_range('2026-01-01','2026-01-31'),n)
})
df['dt']=df['dt'].astype(str)
df.to_sql('sales',conn,index=False)

print("自测题1：各产品销售额Top3的用户(排名+分组模式)")
q1="""WITH ranked AS (
  SELECT user_id, product, SUM(amount) AS total,
    ROW_NUMBER() OVER(PARTITION BY product ORDER BY SUM(amount) DESC) AS rn
  FROM sales GROUP BY user_id, product
)
SELECT product, user_id, ROUND(total,0) AS spend, rn
FROM ranked WHERE rn<=3 ORDER BY product, rn"""
print(pd.read_sql(q1,conn))

print("\n自测题2：每个用户的累计消费(累计模式)")
q2="""SELECT user_id, dt, amount,
  SUM(amount) OVER(PARTITION BY user_id ORDER BY dt) AS cum
FROM sales WHERE user_id IN (1,2,3) ORDER BY user_id, dt LIMIT 15"""
print(pd.read_sql(q2,conn))

print("\n自测题3：各产品GMV占比(占比模式)")
q3="""SELECT product,
  ROUND(SUM(amount),0) AS gmv,
  ROUND(SUM(amount)*100.0/SUM(SUM(amount)) OVER(),1) AS pct
FROM sales GROUP BY product ORDER BY gmv DESC"""
print(pd.read_sql(q3,conn))

print("\n自测题4：用户分群——按总消费高/中/低(分组+条件模式)")
q4="""SELECT
  CASE WHEN total>=1000 THEN '高' WHEN total>=300 THEN '中' ELSE '低' END AS tier,
  COUNT(*) AS users, ROUND(AVG(total),0) AS avg_spend
FROM (SELECT user_id, SUM(amount) AS total FROM sales GROUP BY user_id)
GROUP BY tier ORDER BY AVG(total) DESC"""
print(pd.read_sql(q4,conn))

print("\n自测题5：复购率——购买>=2次的用户占比(复购模式)")
q5="""WITH u AS (SELECT user_id, COUNT(*) AS cnt FROM sales GROUP BY user_id)
SELECT COUNT(*) AS total, SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END) AS rpt,
  ROUND(SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS rate
FROM u"""
print(pd.read_sql(q5,conn))

print("\n--- 自测结果 ---")
print("5/5 → 优秀！可以直接进入下半程")
print("4/5 → 良好！个别知识点再巩固一下")
print("3/5 → 及格，但建议重点复习不熟的2题")
print("<3  → 建议W5-W6重点内容再复习一遍再进入下半程")


## ✅ 区块6：W5-W6分级检验总表

### 🟢 初级(W5-W6必须全部掌握，这是面试及格线)
- [ ] 写出5种窗口函数(ROW_NUMBER/RANK/DENSE_RANK/LAG/SUM OVER)
- [ ] 写出4种JOIN(INNER/LEFT/RIGHT/FULL)并画出韦恩图
- [ ] 写出CTE链(WITH ... AS的3层以上线性依赖)
- [ ] 写出次日留存SQL(LEFT JOIN + DATEDIFF + DATE_ADD)
- [ ] 写出连续登录SQL(gap-and-island四步法)
- [ ] 写出GMV占比SQL(SUM(SUM) OVER嵌套模式)
- [ ] 写出复购率SQL(CASE WHEN cnt>=2)
- [ ] 使用EXPLAIN分析查询(关注type/rows/Extra)
- [ ] 辨别5种以上索引失效场景并给出修复方案
- [ ] 完成月度报告6个核心查询(概览/新老/品类/Top10/复购/高质量新用户)

### 🟡 中级(应掌握80%以上，这是合格分析师的标准)
- [ ] 区分LEFT JOIN + WHERE右表条件 = INNER JOIN的陷阱
- [ ] 识别NOT IN vs NOT EXISTS的NULL处理差异
- [ ] 实现RFM三维打分和用户价值分层
- [ ] 实现活动前/中/后三段对比SQL
- [ ] 实现简单的DID效果估算(活动组变化 - 对照组变化)
- [ ] 识别薅羊毛用户(订单金额门槛附近集中)
- [ ] 实现画像交叉分析(价值分层 × 活跃状态 × 品类偏好)
- [ ] 优化COUNT(DISTINCT)为GROUP BY + COUNT(*)
- [ ] 诊断数据倾斜(查看key分布)并说出4种解决方案
- [ ] 分析渠道获客质量(GMV per user by channel)

### 🔴 高级(追求掌握，这是高级分析师的水平)
- [ ] 设计完整画像标签体系(价值/活跃/偏好/行为/生命周期5维度)
- [ ] 设计活动ROI完整计算方案(增量GMV×毛利率-全部成本)
- [ ] 设计分区+索引+倾斜处理的完整优化方案
- [ ] 处理大数据场景的数据倾斜(加盐打散/MapJoin/拆分后UNION ALL)
- [ ] 设计自动化月度报告方案(SQL脚本+Cron调度+BI看板+自动推送)
- [ ] 从SQL分析结果推导出可执行的业务建议，并设计验证方案(AB实验)
- [ ] 建立"报告→发现→建议→行动→验证→优化"的完整运营闭环

### 🎯 面试准备等级自评

| 面试级别 | SQL能力要求 | 典型面试表现 | 你的自评 |
|---------|-----------|-----------|--------|
| 初级分析师 | 能写正确SQL，基础TopN/留存/JOIN | 能答出正确答案 | ___ |
| 中级分析师 | 能设计方案，讨论性能和边界情况 | 主动讨论为什么这样做 | ___ |
| 高级分析师 | 能设计框架，讨论因果推断和实验 | 从业务问题反推分析体系 | ___ |

**如果你的自评在中级以下(<=2)**：建议W5-W6重点内容再复习1-2天。特别是窗口函数、CTE和JOIN陷阱——这三个最容易出错的知识点。
**如果你的自评在中级(=3)**：恭喜！你已具备大厂数据分析师的SQL基本功。下半程的AB实验和因果推断将让你更进一步。
**如果你的自评在高级(>=4)**：你的SQL能力已经超过了80%的面试者。下半程的重点将是实验设计和因果推断——这是数据分析师真正的"高阶技能"。

## 📝 区块7：阶段二总结与下半程寄语

### 补充：阶段二你最大的收获应该是什么？

如果只能用一句话总结W4-W6的收获，那就是：**我可以用SQL独立完成一个完整的业务分析了。**

这不是一句空话——它意味着：
- 接到"帮我看看上个月的数据"时，你不会慌——你知道该查什么、怎么查
- 看到复杂的业务问题，你会先在纸上画数据流图，再用CTE一步步实现
- 写完SQL后，你会习惯性地跑EXPLAIN检查性能
- 给出结果后，你会配上解读和建议，而不是把数字一扔了事

这就是从"会用SQL"到"能用SQL解决真实业务问题"的质变。带着这个能力进入下半程，你已经有足够的底气去挑战更高级的问题了。

### 最后：给自己一个肯定

你已经连续学习了6周，完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下：6周前的你和现在的你，在数据分析能力上有多大的差距？

把这个差距写下来——它是你继续前进的动力。

**下半程见！**


### 补充：阶段二你最大的收获应该是什么？

如果只能用一句话总结W4-W6的收获，那就是：**我可以用SQL独立完成一个完整的业务分析了。**

这不是一句空话——它意味着：
- 接到"帮我看看上个月的数据"时，你不会慌——你知道该查什么、怎么查
- 看到复杂的业务问题，你会先在纸上画数据流图，再用CTE一步步实现
- 写完SQL后，你会习惯性地跑EXPLAIN检查性能
- 给出结果后，你会配上解读和建议，而不是把数字一扔了事

这就是从"会用SQL"到"能用SQL解决真实业务问题"的质变。带着这个能力进入下半程，你已经有足够的底气去挑战更高级的问题了。

### 给自己一个肯定

你已经连续学习了6周，完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下：6周前的你和现在的你，在数据分析能力上有多大的差距？

把这个差距写下来——它是你继续前进的动力。

### W5-W6技能在实际工作中的应用频率

| 技能 | 日常使用频率 | 典型场景 |
|------|-----------|--------|
| 窗口函数(ROW_NUMBER/LAG/SUM OVER) | 每天 | TopN榜单、环比同比、累计指标 |
| CTE链式设计 | 每天 | 任何超过2步的分析需求 |
| LEFT JOIN + 留存计算 | 每周 | 用户留存监控、渠道质量评估 |
| GROUP BY + CASE WHEN聚合 | 每天 | 用户分群、多维交叉分析 |
| EXPLAIN性能诊断 | 每周 | 慢查询排查、新SQL上线前检查 |
| 用户画像标签构建 | 每月 | 月度画像更新、运营活动人群圈选 |
| 活动效果评估 | 每次活动后 | 活动复盘、ROI计算、策略优化 |

这些技能不是你"学完就忘"的考试内容——它们是你每天工作中真正会用到的工具。

**下半程见！**


### 补充：阶段二你最大的收获应该是什么？

如果只能用一句话总结W4-W6的收获，那就是：**我可以用SQL独立完成一个完整的业务分析了。**

这不是一句空话——它意味着：
- 接到"帮我看看上个月的数据"时，你不会慌——你知道该查什么、怎么查
- 看到复杂的业务问题，你会先在纸上画数据流图，再用CTE一步步实现
- 写完SQL后，你会习惯性地跑EXPLAIN检查性能
- 给出结果后，你会配上解读和建议，而不是把数字一扔了事

这就是从"会用SQL"到"能用SQL解决真实业务问题"的质变。带着这个能力进入下半程，你已经有足够的底气去挑战更高级的问题了。

### 给自己一个肯定

你已经连续学习了6周，完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下：6周前的你和现在的你，在数据分析能力上有多大的差距？

把这个差距写下来——它是你继续前进的动力。

### W5-W6技能在实际工作中的应用频率

| 技能 | 日常使用频率 | 典型场景 |
|------|-----------|--------|
| 窗口函数(ROW_NUMBER/LAG/SUM OVER) | 每天 | TopN榜单、环比同比、累计指标 |
| CTE链式设计 | 每天 | 任何超过2步的分析需求 |
| LEFT JOIN + 留存计算 | 每周 | 用户留存监控、渠道质量评估 |
| GROUP BY + CASE WHEN聚合 | 每天 | 用户分群、多维交叉分析 |
| EXPLAIN性能诊断 | 每周 | 慢查询排查、新SQL上线前检查 |
| 用户画像标签构建 | 每月 | 月度画像更新、运营活动人群圈选 |
| 活动效果评估 | 每次活动后 | 活动复盘、ROI计算、策略优化 |

这些技能不是你"学完就忘"的考试内容——它们是你每天工作中真正会用到的工具。

**下半程见！**


### 补充：阶段二你最大的收获应该是什么

如果只能用一句话总结W4-W6的收获那就是我可以用SQL独立完成一个完整的业务分析了。这不是一句空话它意味着接到帮我看看上个月的数据时你不会慌你知道该查什么怎么查。看到复杂的业务问题你会先在纸上画数据流图再用CTE一步步实现。写完SQL后你会习惯性地跑EXPLAIN检查性能。给出结果后你会配上解读和建议而不是把数字一扔了事。这就是从会用SQL到能用SQL解决真实业务问题的质变。带着这个能力进入下半程你已经有足够的底气去挑战更高级的问题了。

### 给自己一个肯定

你已经连续学习了6周完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下6周前的你和现在的你在数据分析能力上有多大的差距把这个差距写下来它是你继续前进的动力。

### W5-W6技能在实际工作中的应用频率

窗口函数ROW_NUMBER LAG SUM OVER每天使用典型场景TopN榜单环比同比累计指标。CTE链式设计每天使用任何超过2步的分析需求。LEFT JOIN加留存计算每周使用用户留存监控渠道质量评估。GROUP BY加CASE WHEN聚合每天使用用户分群多维交叉分析。EXPLAIN性能诊断每周使用慢查询排查新SQL上线前检查。用户画像标签构建每月使用月度画像更新运营活动人群圈选。活动效果评估每次活动后使用活动复盘ROI计算策略优化。这些技能不是你学完就忘的考试内容它们是你每天工作中真正会用到的工具。

**下半程见！**


### 补充：阶段二你最大的收获应该是什么

如果只能用一句话总结W4-W6的收获那就是我可以用SQL独立完成一个完整的业务分析了。这不是一句空话它意味着接到帮我看看上个月的数据时你不会慌你知道该查什么怎么查。看到复杂的业务问题你会先在纸上画数据流图再用CTE一步步实现。写完SQL后你会习惯性地跑EXPLAIN检查性能。给出结果后你会配上解读和建议而不是把数字一扔了事。这就是从会用SQL到能用SQL解决真实业务问题的质变。带着这个能力进入下半程你已经有足够的底气去挑战更高级的问题了。

### 给自己一个肯定

你已经连续学习了6周完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下6周前的你和现在的你在数据分析能力上有多大的差距把这个差距写下来它是你继续前进的动力。

### W5-W6技能在实际工作中的应用频率

窗口函数ROW_NUMBER LAG SUM OVER每天使用典型场景TopN榜单环比同比累计指标。CTE链式设计每天使用任何超过2步的分析需求。LEFT JOIN加留存计算每周使用用户留存监控渠道质量评估。GROUP BY加CASE WHEN聚合每天使用用户分群多维交叉分析。EXPLAIN性能诊断每周使用慢查询排查新SQL上线前检查。用户画像标签构建每月使用月度画像更新运营活动人群圈选。活动效果评估每次活动后使用活动复盘ROI计算策略优化。这些技能不是你学完就忘的考试内容它们是你每天工作中真正会用到的工具。

### 下半程学习的关键心态

上半程你学的是工具和技能这些是硬实力。下半程你学的是实验设计和因果推断这些是思维方式。工具可以在工作中边做边学但思维方式一旦建立就会影响你整个职业生涯。所以在W7-W12不要急于求成而是要真正理解为什么需要随机化为什么相关性不等于因果性为什么统计显著不等于业务显著。这些为什么的答案才是数据分析师最有价值的资产。

**下半程见！**


### 补充：阶段二你最大的收获应该是什么

如果只能用一句话总结W4-W6的收获那就是我可以用SQL独立完成一个完整的业务分析了。这不是一句空话它意味着接到帮我看看上个月的数据时你不会慌你知道该查什么怎么查。看到复杂的业务问题你会先在纸上画数据流图再用CTE一步步实现。写完SQL后你会习惯性地跑EXPLAIN检查性能。给出结果后你会配上解读和建议而不是把数字一扔了事。这就是从会用SQL到能用SQL解决真实业务问题的质变。带着这个能力进入下半程你已经有足够的底气去挑战更高级的问题了。

### 给自己一个肯定

你已经连续学习了6周完成了整个课程的50%。这在任何学习项目里都是不容易的成就。花一分钟回顾一下6周前的你和现在的你在数据分析能力上有多大的差距把这个差距写下来它是你继续前进的动力。

### W5-W6技能在实际工作中的应用频率

窗口函数ROW_NUMBER LAG SUM OVER每天使用典型场景TopN榜单环比同比累计指标。CTE链式设计每天使用任何超过2步的分析需求。LEFT JOIN加留存计算每周使用用户留存监控渠道质量评估。GROUP BY加CASE WHEN聚合每天使用用户分群多维交叉分析。EXPLAIN性能诊断每周使用慢查询排查新SQL上线前检查。用户画像标签构建每月使用月度画像更新运营活动人群圈选。活动效果评估每次活动后使用活动复盘ROI计算策略优化。这些技能不是你学完就忘的考试内容它们是你每天工作中真正会用到的工具。

### 下半程学习的关键心态

上半程你学的是工具和技能这些是硬实力。下半程你学的是实验设计和因果推断这些是思维方式。工具可以在工作中边做边学但思维方式一旦建立就会影响你整个职业生涯。所以在W7-W12不要急于求成而是要真正理解为什么需要随机化为什么相关性不等于因果性为什么统计显著不等于业务显著。这些为什么的答案才是数据分析师最有价值的资产。

**下半程见！**
